In [3]:
# -*- coding: utf-8 -*-
"""
Ibaraki population (population_ibaraki.xlsx) + SSDSE-A-2025 を使った
市町村別人口の多変量時系列パネルデータ構築 & 簡単な回帰モデル

前提ファイル（同じフォルダに置くこと）:
  - population_ibaraki.xlsx  （各年10月1日現在の市町村別人口：茨城県）
  - SSDSE-A-2025.csv        （SSDSE-市区町村 2025）

やっていること:
  1. population_ibaraki.xlsx をロング形式 (city_name, year, population) に変換
  2. SSDSE-A-2025.csv から茨城県の市区町村データを抜き出し、人口構造などの説明変数を作成
  3. 2つを city_name で結合してパネルデータを作成
  4. モデル
       - Model 1: AR(1)   log_pop_t ~ log_pop_{t-1}
       - Model 2: AR+SSDSE log_pop_t ~ log_pop_{t-1} + SSDSE 指標
  5. 予測結果とパネルデータを CSV に出力
"""

import os
import re
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import statsmodels.api as sm


POP_FILE = "population_ibaraki.xlsx"
SSDSE_FILE = "SSDSE-A-2025.csv"


# ======================================================
# ユーティリティ
# ======================================================

def clean_city_name(name: str) -> str:
    """
    市町村名の表記ゆれを軽く吸収するためのクリーニング。
    - 全角・半角のカッコ内を削除: （つくば市） → つくば市
    - 前後の空白削除
    """
    s = str(name)
    # 全角カッコ内を削除
    s = re.sub(r'（.*?）', '', s)
    # 半角カッコ内を削除
    s = re.sub(r'\(.*?\)', '', s)
    return s.strip()


def parse_year(label: str):
    """
    セルのラベルから西暦の年を推定する。
    - '1975', '1980年' など → そのまま西暦
    - 'S50', 'Ｓ50', '昭和50年' など → 1975 (= 1925 + 50)
    - 'H1', '平成1年' など      → 1989 (= 1988 + 1)
    - 'R6', '令和6年' など      → 2024 (= 2018 + 6)
    該当しなければ None を返す。
    """
    s = str(label)

    # 4桁の西暦が入っている場合
    m = re.search(r'(19|20)\d{2}', s)
    if m:
        return int(m.group(0))

    # 昭和 S or Ｓ
    m = re.search(r'[sSＳ](\d+)', s)
    if m:
        return 1925 + int(m.group(1))

    # 平成 H or Ｈ
    m = re.search(r'[hHＨ](\d+)', s)
    if m:
        return 1988 + int(m.group(1))

    # 令和 R or Ｒ
    m = re.search(r'[rRＲ](\d+)', s)
    if m:
        return 2018 + int(m.group(1))

    # 「昭和50年」「平成1年」「令和6年」みたいな漢字パターン
    if "昭和" in s:
        m = re.search(r'(\d+)', s)
        if m:
            return 1925 + int(m.group(1))
    if "平成" in s:
        m = re.search(r'(\d+)', s)
        if m:
            return 1988 + int(m.group(1))
    if "令和" in s:
        m = re.search(r'(\d+)', s)
        if m:
            return 2018 + int(m.group(1))

    return None


# ======================================================
# 1. population_ibaraki.xlsx の読み込み
# ======================================================

def load_population(path: str) -> pd.DataFrame:
    """
    population_ibaraki.xlsx から
      city_name, year, population
    のロング形式 DataFrame を作る。

    想定レイアウト（ユーザーのログから）:
      1行目: タイトル
      2〜4行目: 注記など
      5行目: 年代の行（→ インデックス4）
        - A列: '（合併前の市町村名）' 等
        - B列以降: '昭和５０年', '昭和５１年', ..., '令和６年'
      6行目以降: データ行
        - A列: 市町村名
        - B列以降: 各年人口
    """
    print("=== Loading population data from", path, "===")
    df_raw = pd.read_excel(path, header=None)
    print("Shape of raw population file:", df_raw.shape)
    print("First 8 rows of raw file:")
    print(df_raw.head(8))

    # 5行目（1始まり）＝ インデックス4 をヘッダー行として扱う
    header_row_idx = 4
    header_row = df_raw.iloc[header_row_idx]

    print("\nHeader row (row index 4):")
    print(header_row)

    # 年の列（A列を除く B列=インデックス1 以降）
    year_cols = []
    for j in range(1, df_raw.shape[1]):
        label = header_row.iloc[j]
        year = parse_year(label)
        if year is not None:
            year_cols.append((j, int(year)))

    if not year_cols:
        raise ValueError("人口ファイルから年次列を検出できませんでした。5行目の内容を確認してください。")

    print("\nDetected year columns (col_index -> year):")
    for col_idx, year in year_cols:
        print(f"  col {col_idx}: {header_row.iloc[col_idx]} -> {year}")

    records = []

    # データ行は 6行目（1始まり）＝ インデックス5 から
    data_start_idx = header_row_idx + 1
    for i in range(data_start_idx, df_raw.shape[0]):
        city = df_raw.iloc[i, 0]
        if pd.isna(city):
            continue
        city = str(city).strip()
        if city == "":
            continue
        # 県全体の行など、不要なら除外
        if city == "茨城県":
            continue

        for col_idx, year in year_cols:
            val = df_raw.iloc[i, col_idx]
            if pd.isna(val):
                continue
            try:
                pop_val = float(val)
            except Exception:
                continue
            records.append((city, int(year), pop_val))

    pop_long = pd.DataFrame(records, columns=["city_name", "year", "population"])
    # 市町村名クリーニング（カッコを落とす）
    pop_long["city_name"] = pop_long["city_name"].map(clean_city_name)

    pop_long = pop_long.sort_values(["city_name", "year"]).reset_index(drop=True)

    print("\nPopulation long-format sample:")
    print(pop_long.head())

    return pop_long


# ======================================================
# 2. SSDSE-A-2025.csv の読み込み（茨城県だけ抽出）
# ======================================================

def load_ssdse(path: str):
    """
    SSDSE-A-2025.csv から、茨城県の市区町村データだけを抜き出して返す。

    想定構造:
      1行目: SSDSE-A-2025, Prefecture, Municipality, A1101, ...
      2行目: , , 年度, 2020, 2020, ...
      3行目: 地域コード, 都道府県, 市区町村, 総人口, ...
      4行目以降: 各市区町村のデータ

    → 1行目をヘッダーとして使い、Prefecture == '茨城県' の行を抽出。
    """
    print("\n=== Loading SSDSE data from", path, "===")

    # 1) ヘッダー = 1行目（0-index）として読み込む
    for enc in ["cp932", "utf-8-sig", "utf-8"]:
        try:
            ssdse_raw = pd.read_csv(path, encoding=enc, header=0)
            print("  -> Loaded SSDSE with encoding:", enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise RuntimeError("SSDSE-A-2025.csv を読み込めませんでした（文字コードの問題）。")

    print("Columns in SSDSE file (first 15):")
    print(list(ssdse_raw.columns)[:15])

    cols = list(ssdse_raw.columns)

    # Prefecture / Municipality 列を特定
    if "Prefecture" in cols:
        pref_col = "Prefecture"
    elif "都道府県" in cols:
        pref_col = "都道府県"
    else:
        raise KeyError("SSDSE に 'Prefecture' または '都道府県' 列が見つかりません。")

    if "Municipality" in cols:
        muni_col = "Municipality"
    elif "市区町村" in cols:
        muni_col = "市区町村"
    else:
        raise KeyError("SSDSE に 'Municipality' または '市区町村' 列が見つかりません。")

    # 地域コード（あれば city_code に）
    if "地域コード" in cols:
        code_col = "地域コード"
    elif "AreaCode" in cols:
        code_col = "AreaCode"
    else:
        code_col = None

    # 茨城県だけ抽出（Prefecture 列は '茨城県' が入っているはず）
    ssdse_ibaraki = ssdse_raw[ssdse_raw[pref_col] == "茨城県"].copy()
    ssdse_ibaraki["city_name"] = ssdse_ibaraki[muni_col].astype(str).map(clean_city_name)

    if code_col is not None:
        ssdse_ibaraki["city_code"] = (
            ssdse_ibaraki[code_col].astype(str).str.replace("^R", "", regex=True)
        )
    else:
        ssdse_ibaraki["city_code"] = np.nan

    print("\nSSDSE Ibaraki sample:")
    print(ssdse_ibaraki[[pref_col, muni_col]].head())

    # 使いたい SSDSE 項目コード（1行目のコード名）
    # 必要に応じて増減してOK
    candidate_x_cols = [
        "A1301",   # 15歳未満人口
        "A1302",   # 15～64歳人口
        "A1303",   # 65歳以上人口
        "A4101",   # 出生数
        "A4200",   # 死亡数
        "A5101",   # 転入者数（日本人移動者）
        "A5102",   # 転出者数（日本人移動者）
        "A7101",   # 世帯数
        "A710101", # 一般世帯数
    ]
    x_cols = [c for c in candidate_x_cols if c in ssdse_ibaraki.columns]

    if not x_cols:
        print("WARNING: 指定した SSDSE 項目コードが1つも見つかりませんでした。")
        print("SSDSE-A-2025.csv の1行目（コード行）と candidate_x_cols を見比べて修正してください。")
    else:
        print("Using SSDSE columns:", x_cols)

    # 数値に変換（文字列が混ざっていても NaN に落とす）
    for c in x_cols:
        ssdse_ibaraki[c] = pd.to_numeric(ssdse_ibaraki[c], errors="coerce")

    cols_to_keep = ["city_name", "city_code"] + x_cols
    ssdse_sub = ssdse_ibaraki[cols_to_keep].copy().drop_duplicates(subset=["city_name"])

    print("\nSSDSE sub sample:")
    print(ssdse_sub.head())

    return ssdse_sub, x_cols


# ======================================================
# 3. パネルデータの構築
# ======================================================

def build_panel(pop_df: pd.DataFrame, ssdse_df: pd.DataFrame, x_cols):
    """
    population（city_name, year, population）と
    SSDSE（city_name, ...）を結合してパネルデータを作る。
    """
    print("\n=== Building panel dataset ===")

    pop_df = pop_df.copy()
    # 念のため city_name をクリーニング
    pop_df["city_name"] = pop_df["city_name"].map(clean_city_name)
    ssdse_df = ssdse_df.copy()
    ssdse_df["city_name"] = ssdse_df["city_name"].map(clean_city_name)

    pop_df["log_pop"] = np.log(pop_df["population"])

    pop_df = pop_df.sort_values(["city_name", "year"])
    pop_df["log_pop_lag1"] = pop_df.groupby("city_name")["log_pop"].shift(1)

    panel = pop_df.dropna(subset=["log_pop_lag1"]).copy()

    panel = panel.merge(
        ssdse_df,
        on="city_name",
        how="left",
        suffixes=("", "_ssdse")
    )

    print("Panel sample (before dropping missing SSDSE):")
    print(panel.head())

    if x_cols:
        before = len(panel)
        panel = panel.dropna(subset=x_cols)
        after = len(panel)
        print(f"Dropped {before - after} rows due to missing SSDSE features.")
    else:
        print("No SSDSE features found; AR(1) モデルのみ実行します。")

    print("\nPanel sample (after dropping missing SSDSE):")
    print(panel.head())

    return panel


# ======================================================
# 4. モデル推定（AR と AR+SSDSE）
# ======================================================

def fit_models(panel: pd.DataFrame, x_cols):
    """
    2つのモデルを推定して比較:
      1) AR(1): log_pop_t ~ log_pop_{t-1}
      2) AR(1)+SSDSE: log_pop_t ~ log_pop_{t-1} + SSDSE指標
    """
    print("\n=== Fitting models ===")

    panel = panel.copy()

    if x_cols:
        scaler = StandardScaler()
        panel[x_cols] = scaler.fit_transform(panel[x_cols])

    panel["const"] = 1.0
    y = panel["log_pop"]
    X_ar = panel[["const", "log_pop_lag1"]]
    X_full = panel[["const", "log_pop_lag1"] + x_cols] if x_cols else None

    # 学習: 2015年まで / テスト: 2016年以降
    train_mask = panel["year"] <= 2015
    test_mask = panel["year"] > 2015

    # ---- モデル1: AR(1) ----
    print("\n===== Model 1: AR(1) only =====")
    model_ar = sm.OLS(y[train_mask], X_ar[train_mask]).fit()
    print(model_ar.summary())

    y_pred_log_ar = model_ar.predict(X_ar[test_mask])
    y_pred_ar = np.exp(y_pred_log_ar)
    y_true = np.exp(y[test_mask])

    mape_ar = mean_absolute_percentage_error(y_true, y_pred_ar)
    # ★ sklearn 古いので squared=False は使わず、自分で RMSE を計算 ★
    mse_ar = mean_squared_error(y_true, y_pred_ar)
    rmse_ar = np.sqrt(mse_ar)

    print(f"\nAR(1) MAPE (test): {mape_ar:.4f}")
    print(f"AR(1) RMSE  (test): {rmse_ar:.2f}")

    result_ar = panel.loc[test_mask, ["city_name", "year"]].copy()
    result_ar["pop_true"] = y_true.values
    result_ar["pop_pred_ar"] = y_pred_ar
    result_ar.to_csv("predictions_ar.csv", index=False, encoding="utf-8-sig")
    print("Saved predictions_ar.csv")

    # ---- モデル2: AR(1) + SSDSE ----
    if X_full is not None and len(x_cols) > 0:
        print("\n===== Model 2: AR(1) + SSDSE =====")
        model_full = sm.OLS(y[train_mask], X_full[train_mask]).fit()
        print(model_full.summary())

        y_pred_log_full = model_full.predict(X_full[test_mask])
        y_pred_full = np.exp(y_pred_log_full)

        mape_full = mean_absolute_percentage_error(y_true, y_pred_full)
        mse_full = mean_squared_error(y_true, y_pred_full)
        rmse_full = np.sqrt(mse_full)

        print(f"\nAR(1)+SSDSE MAPE (test): {mape_full:.4f}")
        print(f"AR(1)+SSDSE RMSE  (test): {rmse_full:.2f}")

        result_full = result_ar.copy()
        result_full["pop_pred_full"] = y_pred_full
        result_full.to_csv("predictions_ar_ssdse.csv", index=False, encoding="utf-8-sig")
        print("Saved predictions_ar_ssdse.csv")
    else:
        print("\nSSDSE 説明変数がないため、AR(1) モデルのみ推定しました。")


# ======================================================
# 5. メイン
# ======================================================

def main():
    # ファイル存在チェック
    if not os.path.exists(POP_FILE):
        raise FileNotFoundError(
            f"{POP_FILE} が見つかりません。\n"
            f"この .py ファイルと同じフォルダに population_ibaraki.xlsx を置くか、"
            f"POP_FILE を正しいファイル名に直してください。"
        )
    if not os.path.exists(SSDSE_FILE):
        raise FileNotFoundError(
            f"{SSDSE_FILE} が見つかりません。\n"
            f"この .py ファイルと同じフォルダに SSDSE-A-2025.csv を置くか、"
            f"SSDSE_FILE を正しいファイル名に直してください。"
        )

    # 1) 人口データ
    pop_df = load_population(POP_FILE)

    # 2) SSDSE データ
    ssdse_df, x_cols = load_ssdse(SSDSE_FILE)

    # 3) パネル構築
    panel = build_panel(pop_df, ssdse_df, x_cols)

    # 4) モデル推定
    fit_models(panel, x_cols)

    # 5) パネル全体を保存（可視化などに使える）
    panel.to_csv("panel_ibaraki_ssdse.csv", index=False, encoding="utf-8-sig")
    print("\nSaved panel_ibaraki_ssdse.csv")


if __name__ == "__main__":
    main()


=== Loading population data from population_ibaraki.xlsx ===
Shape of raw population file: (161, 52)
First 8 rows of raw file:
                                           0             1   \
0  市町村別人口の推移（平成１８年４月行政区画（４４市町村）による） 各年１０月１日現在           NaN   
1                                         NaN           NaN   
2                                         NaN           NaN   
3                                        市町村名  平成12年以降の合併期日   
4                                  （合併前の市町村名）           NaN   
5                                         NaN           NaN   
6                                         茨城県           NaN   
7                                         NaN           NaN   

                                                  2        3        4   \
0                                                NaN      NaN      NaN   
1                                                NaN      NaN      NaN   
2  昭和５０、５５、６０、平成２、７、１２、１７、２２年、２７、令和２年は国勢調査結果、その他の...      NaN      NaN   
3        

In [4]:
import pandas as pd
import numpy as np

# 予測結果を読み込み
ar = pd.read_csv("predictions_ar.csv")
full = pd.read_csv("predictions_ar_ssdse.csv")

# 両方を結合
df = ar.merge(full[["city_name", "year", "pop_pred_full"]],
              on=["city_name", "year"],
              how="left")

# 絶対誤差・パーセント誤差
df["abs_err_ar"] = (df["pop_pred_ar"] - df["pop_true"]).abs()
df["abs_err_full"] = (df["pop_pred_full"] - df["pop_true"]).abs()

df["ape_ar"] = df["abs_err_ar"] / df["pop_true"]
df["ape_full"] = df["abs_err_full"] / df["pop_true"]

# 市町村ごとに平均 APE を計算
by_city = (
    df.groupby("city_name")[["ape_ar", "ape_full"]]
      .mean()
      .reset_index()
)

by_city["delta_ape"] = by_city["ape_full"] - by_city["ape_ar"]  # マイナスなら SSDSE モデルの方が良い

# SSDSEモデルの方が改善している市町村トップ10
improved = by_city.sort_values("delta_ape").head(10)
print("SSDSE を入れた方が誤差が小さい市町村 TOP10:")
print(improved)

# 悪化している市町村トップ10
worse = by_city.sort_values("delta_ape", ascending=False).head(10)
print("\nSSDSE を入れた方が誤差が大きい市町村 TOP10:")
print(worse)


SSDSE を入れた方が誤差が小さい市町村 TOP10:
   city_name    ape_ar  ape_full  delta_ape
22       日立市  0.021018  0.009196  -0.011822
29       石岡市  0.016858  0.005432  -0.011427
20     常陸大宮市  0.024259  0.014309  -0.009949
39       鉾田市  0.014844  0.005314  -0.009529
8       北茨城市  0.021035  0.011757  -0.009278
2       つくば市  0.011531  0.002991  -0.008541
21     常陸太田市  0.024804  0.016883  -0.007920
37       行方市  0.024512  0.018339  -0.006173
31       稲敷市  0.026154  0.020311  -0.005843
41       高萩市  0.023474  0.018860  -0.004614

SSDSE を入れた方が誤差が大きい市町村 TOP10:
   city_name    ape_ar  ape_full  delta_ape
28       牛久市  0.006237  0.027336   0.021099
5        五霞町  0.022345  0.034049   0.011704
3     ひたちなか市  0.007338  0.018735   0.011398
7        利根町  0.017049  0.025056   0.008006
17       守谷市  0.005301  0.012575   0.007273
25       水戸市  0.007015  0.014184   0.007169
9        取手市  0.008650  0.015461   0.006811
26       河内町  0.031561  0.038142   0.006581
10       古河市  0.008639  0.014264   0.005624
35       美浦村  0.0